# Hand Detection Training - ProjectGrafika
Training YOLOv8 model untuk deteksi tangan menggunakan dataset hand-detection-dataset (VOC/YOLO format).

## Struktur Dataset
```
hand-detection-dataset-vocyolo-format/
├── train/
│   ├── images/   ← gambar training
│   └── labels/   ← anotasi YOLO format
└── test/
    ├── images/   ← gambar testing
    └── labels/   ← anotasi YOLO format
```

## 1. Import Library

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import os
import shutil
from pathlib import Path

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
print(f'OpenCV version   : {cv2.__version__}')

## 2. Install Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

## 3. Setup Dataset
Sesuaikan `DATASET_ROOT` dengan lokasi dataset di komputer kamu.

In [ ]:
# ── Konfigurasi path dataset ──────────────────────────────────────────────────
# Ganti path ini sesuai lokasi dataset kamu:
#   - Kaggle  : /kaggle/input/hand-detection-dataset-vocyolo-format
#   - Lokal   : ./dataset/hand-detection-dataset-vocyolo-format

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    DATASET_SRC  = '/kaggle/input/hand-detection-dataset-vocyolo-format'
    DATASET_ROOT = '/kaggle/working/hand-detection-dataset-vocyolo-format'
    OUTPUT_DIR   = '/kaggle/working/runs'
else:
    # Lokal — dataset di sebelah notebook ini
    DATASET_SRC  = str(Path('./dataset/hand-detection-dataset-vocyolo-format').resolve())
    DATASET_ROOT = DATASET_SRC
    OUTPUT_DIR   = str(Path('./runs').resolve())

print(f'Running on       : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Dataset source   : {DATASET_SRC}')
print(f'Dataset working  : {DATASET_ROOT}')
print(f'Output dir       : {OUTPUT_DIR}')

In [ ]:
# ── Copy dataset ke working dir (Kaggle only) ─────────────────────────────────
if IS_KAGGLE:
    if not os.path.exists(DATASET_ROOT):
        shutil.copytree(DATASET_SRC, DATASET_ROOT)
        print('Dataset copied to working dir')
    else:
        print('Dataset already exists in working dir')

# ── Verifikasi struktur dataset ───────────────────────────────────────────────
for split in ['train', 'test']:
    img_dir = Path(DATASET_ROOT) / split / 'images'
    lbl_dir = Path(DATASET_ROOT) / split / 'labels'
    n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'{split:6s} → images: {n_img:4d} | labels: {n_lbl:4d}')

## 4. Siapkan Label (YOLO Format)

In [ ]:
# Hapus folder VOC (tidak dipakai), pindahkan label YOLO ke folder utama
for split in ['train', 'test']:
    voc_dir  = Path(DATASET_ROOT) / split / 'labels' / 'VOC'
    yolo_dir = Path(DATASET_ROOT) / split / 'labels' / 'YOLO'
    lbl_dir  = Path(DATASET_ROOT) / split / 'labels'

    # Hapus folder VOC
    if voc_dir.exists():
        shutil.rmtree(voc_dir)
        print(f'Removed: {voc_dir}')

    # Pindahkan label YOLO ke parent
    if yolo_dir.exists():
        for f in yolo_dir.glob('*.txt'):
            shutil.move(str(f), str(lbl_dir / f.name))
        yolo_dir.rmdir()
        print(f'Moved YOLO labels for {split}')
    else:
        print(f'YOLO labels for {split} already in place')

print('\nLabel setup selesai!')

## 5. Buat File Konfigurasi Dataset (data.yaml)

In [ ]:
yaml_content = f"""# Hand Detection Dataset Configuration
# ProjectGrafika - Hand Tracking Drawing System

path: {DATASET_ROOT}
train: train/images
val: test/images
test: test/images

nc: 1
names:
  0: hand
"""

yaml_path = Path(DATASET_ROOT) / 'data.yaml'
yaml_path.write_text(yaml_content)

print(f'data.yaml saved to: {yaml_path}')
print('\nIsi data.yaml:')
print(yaml_content)

## 6. Eksplorasi Dataset (EDA)

In [ ]:
# Tampilkan beberapa sampel gambar training beserta bounding box
def show_samples(split='train', n=6):
    img_dir = Path(DATASET_ROOT) / split / 'images'
    lbl_dir = Path(DATASET_ROOT) / split / 'labels'
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    imgs = imgs[:n]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'Sampel Dataset - {split.capitalize()}', fontsize=14, fontweight='bold')

    for ax, img_path in zip(axes.flatten(), imgs):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = map(float, parts)
                        x1 = int((cx - bw/2) * w)
                        y1 = int((cy - bh/2) * h)
                        x2 = int((cx + bw/2) * w)
                        y2 = int((cy + bh/2) * h)
                        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                        cv2.putText(img, 'hand', (x1, y1-5),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_samples('train', 6)

In [ ]:
# Statistik dataset
def dataset_stats():
    stats = {}
    for split in ['train', 'test']:
        img_dir = Path(DATASET_ROOT) / split / 'images'
        lbl_dir = Path(DATASET_ROOT) / split / 'labels'
        imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        
        total_hands = 0
        widths, heights = [], []
        for lbl in lbl_dir.glob('*.txt'):
            lines = lbl.read_text().strip().split('\n')
            total_hands += len([l for l in lines if l.strip()])
        
        # Ambil ukuran gambar dari beberapa sampel
        for img_path in imgs[:20]:
            img = cv2.imread(str(img_path))
            if img is not None:
                h, w = img.shape[:2]
                widths.append(w)
                heights.append(h)
        
        stats[split] = {
            'images': len(imgs),
            'total_hands': total_hands,
            'avg_width': int(np.mean(widths)) if widths else 0,
            'avg_height': int(np.mean(heights)) if heights else 0,
        }
    
    df = pd.DataFrame(stats).T
    print('=== Statistik Dataset ===')
    print(df.to_string())
    return df

stats_df = dataset_stats()

## 7. Training YOLOv8

In [ ]:
# ── Konfigurasi training ───────────────────────────────────────────────────────
MODEL_SIZE  = 'yolov8m.pt'   # nano=n, small=s, medium=m, large=l, xlarge=x
EPOCHS      = 50
IMG_SIZE    = 640
BATCH_SIZE  = 16             # Kurangi ke 8 jika VRAM tidak cukup
PATIENCE    = 20             # Early stopping
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'
PROJECT_NAME = 'ProjectGrafika-HandDetection'

print(f'Model    : {MODEL_SIZE}')
print(f'Epochs   : {EPOCHS}')
print(f'Img size : {IMG_SIZE}')
print(f'Batch    : {BATCH_SIZE}')
print(f'Device   : {DEVICE}')
print(f'data.yaml: {yaml_path}')

In [ ]:
# ── Mulai training ─────────────────────────────────────────────────────────────
model = YOLO(MODEL_SIZE)

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    patience= PATIENCE,
    project = OUTPUT_DIR,
    name    = PROJECT_NAME,
    save    = True,
    plots   = True,
    verbose = True,
)

print('\nTraining selesai!')
print(f'Model tersimpan di: {OUTPUT_DIR}/{PROJECT_NAME}')

## 8. Evaluasi Model

In [ ]:
# Load model terbaik hasil training
best_model_path = Path(OUTPUT_DIR) / PROJECT_NAME / 'weights' / 'best.pt'
print(f'Loading best model dari: {best_model_path}')

best_model = YOLO(str(best_model_path))

# Evaluasi pada test set
metrics = best_model.val(
    data   = str(yaml_path),
    split  = 'test',
    device = DEVICE,
    imgsz  = IMG_SIZE,
)

print(f'\n=== Hasil Evaluasi ===')
print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.p[0] if len(metrics.box.p) else 0:.4f}')
print(f'Recall    : {metrics.box.r[0] if len(metrics.box.r) else 0:.4f}')

## 9. Visualisasi Hasil Training

In [ ]:
# Tampilkan grafik training (loss & metrics)
results_dir = Path(OUTPUT_DIR) / PROJECT_NAME
plot_files = [
    ('results.png',       'Training Curves'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('val_batch0_pred.jpg',  'Validation Predictions'),
]

for filename, title in plot_files:
    fpath = results_dir / filename
    if fpath.exists():
        img = mpimg.imread(str(fpath))
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.title(title, fontsize=13, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'File tidak ditemukan: {fpath}')

## 10. Inferensi pada Gambar Test

In [ ]:
# Jalankan inferensi pada beberapa gambar test
def run_inference(model, split='test', n=6, conf=0.5):
    img_dir = Path(DATASET_ROOT) / split / 'images'
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    imgs = imgs[:n]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'Hasil Deteksi Tangan - {split.capitalize()}', fontsize=14, fontweight='bold')

    for ax, img_path in zip(axes.flatten(), imgs):
        result = model.predict(str(img_path), conf=conf, verbose=False)[0]
        img = result.plot()  # gambar dengan bounding box
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        n_det = len(result.boxes)
        ax.imshow(img)
        ax.set_title(f'{img_path.name} ({n_det} tangan)', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

run_inference(best_model, 'test', n=6, conf=0.5)

## 11. Export Model untuk Integrasi Java (ProjectGrafika)
Export model ke format ONNX agar bisa dipakai di Java dengan OpenCV DNN.

In [ ]:
# Export ke ONNX
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)
print(f'Model ONNX tersimpan di: {onnx_path}')
print()
print('=== Cara Integrasi ke ProjectGrafika (Java) ===')
print('1. Copy file best.onnx ke folder ProjectGrafika/lib/')
print('2. Di ImageProcessor.java, load model dengan:')
print('   Net net = Dnn.readNetFromONNX("lib/best.onnx");')
print('3. Gunakan net.forward() untuk inferensi per frame')

In [ ]:
# Ringkasan akhir
print('=' * 50)
print('  TRAINING SELESAI - ProjectGrafika')
print('=' * 50)
print(f'Best model  : {best_model_path}')
print(f'ONNX model  : {onnx_path}')
print(f'mAP50       : {metrics.box.map50:.4f}')
print(f'mAP50-95    : {metrics.box.map:.4f}')
print('=' * 50)